# My first Metaflow flow with data and decisions

I worked through Metaflow's basic flow concepts — steps, parameters, branching, and data artifacts. The goal: build a flow that loads data, processes it, makes a decision based on a quality threshold, then reports the outcome.

In [ ]:
from metaflow import FlowSpec, step, Parameter, current

## Step 1 — Define the flow

I wrote a simple three-stage flow:
- **load_data**: generate some fake data (pretend it's a CSV read)
- **check_quality**: calculate a quality score and decide
- **report**: print the result

In [ ]:
class DataDecisionFlow(FlowSpec):

    threshold = Parameter(
        "threshold",
        help="Minimum quality score to pass",
        default=0.7
    )

    @step
    def start(self):
        # Load some pretend data — later I'll swap this with a real CSV
        self.records = [
            {"id": i, "value": i * 1.5, "quality": 0.6 + (i * 0.05)}
            for i in range(5)
        ]
        print(f"Loaded {len(self.records)} records")
        self.next(self.check_quality)

    @step
    def check_quality(self):
        # Calculate average quality across all records
        self.avg_quality = sum(r["quality"] for r in self.records) / len(self.records)
        print(f"Average quality: {self.avg_quality:.2f}")

        # Decision: if quality is good enough, process; otherwise flag
        if self.avg_quality >= self.threshold:
            self.next(self.process_high_quality)
        else:
            self.next(self.process_low_quality)

    @step
    def process_high_quality(self):
        self.result = "PASS — data quality is acceptable"
        print(self.result)
        self.next(self.end)

    @step
    def process_low_quality(self):
        self.result = "FAIL — data quality below threshold, need cleaning"
        print(self.result)
        self.next(self.end)

    @step
    def end(self):
        print(f"Flow complete. Final verdict: {self.result}")

## Step 2 — Run the flow

Metaflow runs the flow as a subprocess. I ran it twice — once with the default threshold and once with a high threshold to trigger the low-quality branch.

In [ ]:
if __name__ == "__main__":
    DataDecisionFlow()

### Run with a higher threshold

```bash
python 2026-05-28-first-end-to-end-flow-with-data.ipynb run --threshold 0.9
```

Expected output:
```
Loaded 5 records
Average quality: 0.70
FAIL — data quality below threshold, need cleaning
Flow complete. Final verdict: FAIL — data quality below threshold, need cleaning
```

## What tripped me up

- **The flow must be in a standalone `.py` file**, not run inline in a notebook cell. Metaflow spawns a subprocess and looks for the class in a file. I got `FlowNotFoundError` the first time because I tried to define and run the flow in the same notebook cell. Fix: write the flow to a `.py` file and run it with `!python flow.py run`.
- **`self.next()` after a decision branch** — when you call `self.next(self.step_a, self.step_b)`, both branches must converge or each must have its own `self.next()`. I originally tried to branch and then merge without both branches calling `next`, and Metaflow complained about orphaned steps.
- **Parameter values are strings from the CLI** — `Parameter("threshold", default=0.7)` works for the default, but passing `--threshold 0.9` on the command line gives a string `"0.9"`. Metaflow converts it to float automatically for numeric defaults, but if the default is `None` or a string, you get a string back. I need to remember this for custom types.

## What I'd try next

Now that I have a basic branching flow working, I want to add a real CSV load step and split data into train/test sets based on a condition. After that, I'll try Metaflow's built-in `@resources` decorator to request a GPU — mostly to see if the local scheduler respects it.